# Atividade 3 - Algoritmos de Classificação e Agrupamento
**Disciplina:** Técnicas e Algoritmos em Ciência de Dados  
**Aluno:** Carlos Eduardo Borba Domingues  

Neste notebook serão aplicadas técnicas de:
- Classificação: KNN, Árvore de Decisão e Random Forest
- Agrupamento: K-Means
- Redução de dimensionalidade: PCA

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, silhouette_score
)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('Bibliotecas importadas com sucesso!')

## 2. Carregamento e Exploração dos Dados

In [ ]:
# Carregando o dataset Iris (clássico para classificação)
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print('Shape:', df.shape)
print('\nClasses:', df['species'].value_counts().to_dict())
df.head()

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Visualização: Pairplot
feature_cols = iris.feature_names
sns.pairplot(df, vars=feature_cols, hue='species', palette='Set1')
plt.suptitle('Pairplot - Dataset Iris', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlação
plt.figure(figsize=(8, 6))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Matriz de Correlação - Iris')
plt.tight_layout()
plt.show()

## 3. Pré-processamento

In [ ]:
# Separando features e target
X = df[feature_cols].values
y = df['target'].values

# Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

## 4. Algoritmo KNN (K-Nearest Neighbors)

In [ ]:
# Testando diferentes valores de K
k_values = range(1, 21)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_scaled, y, cv=5, scoring='accuracy')
    accuracies.append(scores.mean())

best_k = k_values[np.argmax(accuracies)]
print(f'Melhor K: {best_k} com acurácia de {max(accuracies):.4f}')

plt.figure(figsize=(10, 5))
plt.plot(k_values, accuracies, 'bo-', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Melhor K={best_k}')
plt.title('Acurácia KNN por valor de K (Cross-Validation)')
plt.xlabel('Número de Vizinhos (K)')
plt.ylabel('Acurácia Média')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Treinamento com o melhor K
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
y_pred_knn = knn_best.predict(X_test)

print('=== KNN - Relatório de Classificação ===')
print(classification_report(y_test, y_pred_knn,
      target_names=iris.target_names))

# Matriz de confusão
plt.figure(figsize=(7, 5))
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title(f'Matriz de Confusão - KNN (K={best_k})')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.tight_layout()
plt.show()

## 5. Árvore de Decisão

In [ ]:
# Treinamento da Árvore de Decisão
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('=== Árvore de Decisão - Relatório de Classificação ===')
print(classification_report(y_test, y_pred_dt,
      target_names=iris.target_names))

In [ ]:
# Visualização da árvore
plt.figure(figsize=(20, 8))
plot_tree(dt, feature_names=feature_cols,
          class_names=iris.target_names,
          filled=True, rounded=True, fontsize=10)
plt.title('Árvore de Decisão - Dataset Iris')
plt.tight_layout()
plt.show()

In [ ]:
# Importância das features
feat_importance = pd.Series(dt.feature_importances_,
                             index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=feat_importance.values, y=feat_importance.index, palette='viridis')
plt.title('Importância das Features - Árvore de Decisão')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

print(feat_importance)

## 6. Random Forest

In [ ]:
# Treinamento do Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=4,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest - Relatório de Classificação ===')
print(classification_report(y_test, y_pred_rf,
      target_names=iris.target_names))

In [ ]:
# Comparativo de acurácia dos modelos
models = {
    'KNN': accuracy_score(y_test, y_pred_knn),
    'Árvore de Decisão': accuracy_score(y_test, y_pred_dt),
    'Random Forest': accuracy_score(y_test, y_pred_rf)
}

plt.figure(figsize=(9, 5))
bars = plt.bar(models.keys(), models.values(),
               color=['steelblue', 'lightcoral', 'mediumseagreen'],
               edgecolor='black', width=0.5)

for bar, val in zip(bars, models.values()):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.ylim(0.8, 1.05)
plt.title('Comparativo de Acurácia dos Modelos de Classificação')
plt.ylabel('Acurácia')
plt.tight_layout()
plt.show()

print('\nAcurácias:')
for m, a in models.items():
    print(f'  {m}: {a:.4f}')

## 7. Redução de Dimensionalidade com PCA

In [ ]:
# Aplicando PCA com 2 componentes
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Variância explicada por componente: {pca.explained_variance_ratio_}')
print(f'Variância total explicada: {pca.explained_variance_ratio_.sum():.4f}')

# Visualização PCA
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['species'] = df['species'].values

plt.figure(figsize=(10, 7))
colors = {'setosa': 'steelblue', 'versicolor': 'lightcoral', 'virginica': 'mediumseagreen'}
for sp, grp in pca_df.groupby('species'):
    plt.scatter(grp['PC1'], grp['PC2'], label=sp,
                color=colors[sp], alpha=0.7, s=60)

plt.title(f'PCA - 2 Componentes (Variância explicada: {pca.explained_variance_ratio_.sum():.2%})')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Agrupamento com K-Means

In [ ]:
# Método Elbow para escolha do K ideal
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2)
axes[0].set_title('Método Elbow - Inércia')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia')
axes[0].grid(True, alpha=0.4)

axes[1].plot(k_range, silhouettes, 'ro-', linewidth=2)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

best_sil_k = k_range[np.argmax(silhouettes)]
print(f'Melhor K pelo Silhouette Score: {best_sil_k}')

In [ ]:
# Aplicando K-Means com K=3 (sabemos que são 3 espécies)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Visualização dos clusters no espaço PCA
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=clusters, cmap='Set1', alpha=0.7, s=60)
centers_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            c='black', s=200, marker='X', label='Centróides', zorder=5)
plt.title('K-Means (K=3) - Clusters')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()

plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                       c=y, cmap='Set1', alpha=0.7, s=60)
plt.title('Classes Reais (para comparação)')
plt.xlabel('PC1')
plt.ylabel('PC2')

plt.tight_layout()
plt.show()

sil = silhouette_score(X_scaled, clusters)
print(f'Silhouette Score (K=3): {sil:.4f}')

## 9. Conclusões

### Classificação:
- **KNN**: Algoritmo simples e eficaz. A escolha do K ideal via cross-validation melhorou o desempenho.
- **Árvore de Decisão**: Facilmente interpretável; as features de pétala foram as mais importantes.
- **Random Forest**: Geralmente o mais robusto dos três por combinar múltiplas árvores (ensemble).

### PCA:
- Dois componentes principais capturam a maior parte da variância do dataset, permitindo boa separação visual das classes.

### K-Means:
- O agrupamento com K=3 recuperou bem os grupos naturais do dataset, confirmando a estrutura conhecida das espécies de Iris.
- O Silhouette Score indica a coesão e separação dos clusters formados.